# ClimateTwin — Student Baseline Notebook

**AquaAir General Insurance | Commercial Property & BI Environmental Impairment Case**

This notebook accompanies `case_study.md`, `dataset.csv`, and `data_dictionary.md`.

### Decision cycle
**Observe → Model → Price → Stress → Mitigate → Reprice**

### Important rule
A poor air- or water-quality reading is **not itself a claim**. Your work must preserve the chain from environmental hazard to qualifying insured event to insurer-paid loss.

Complete the cells marked **TODO**. Keep the core case limited to the four required scenarios; advanced copulas, spatial models, reinsurance, and adaptive optimization are outside the required exercise.


## 0. Setup and data loading

Run the next cells first. The dataset should contain 750 policy-location-year rows and 250 unique locations.

In [ ]:
# Core imports and reproducibility
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

SEED = 20260908
rng = np.random.default_rng(SEED)

DATA_PATH = Path("dataset.csv")
FIXED_EXPENSE = 750
VARIABLE_EXPENSE_RATIO = 0.25
PROFIT_CONTINGENCY = 0.05
RESILIENCE_BUDGET = 8_000_000

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


In [ ]:
# Load the exact teaching dataset
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Years:", sorted(df["year"].unique()))
print("Locations:", df["location_id"].nunique())
print("Claims:", int(df["claim_count"].sum()))
display(df.head())


In [ ]:
# Helper feature engineering used throughout the case
def add_model_features(data):
    x = data.copy()
    x["pm25_10"] = x["pm25"] / 10.0
    x["ozone_10"] = x["ozone"] / 10.0
    x["turbidity_2"] = x["turbidity"] / 2.0
    x["wqi_10"] = x["water_quality_index"] / 10.0
    x["temp_5"] = x["temperature_c"] / 5.0
    x["rain_500"] = x["annual_rainfall_mm"] / 500.0
    x["log_iv"] = np.log(x["insured_value"])
    x["air_water_interaction"] = x["air_stress_flag"] * x["water_stress_flag"]
    return x

def apply_scenario(data, scenario):
    x = data.copy()

    if scenario in ("Air stress", "Compound stress"):
        x["pm25"] = x["pm25"] + 15.0
        x["ozone"] = x["ozone"] + 10.0

    if scenario in ("Water stress", "Compound stress"):
        x["turbidity"] = x["turbidity"] * 1.60
        x["water_quality_index"] = (x["water_quality_index"] - 15.0).clip(lower=0)

    x["air_stress_flag"] = ((x["pm25"] >= 35.0) | (x["ozone"] >= 70.0)).astype(int)
    x["water_stress_flag"] = ((x["turbidity"] >= 5.0) | (x["water_quality_index"] <= 60.0)).astype(int)

    return add_model_features(x)

dfm = add_model_features(df)
portfolio_2026 = dfm.loc[dfm["year"] == 2026].copy().reset_index(drop=True)


## Stage 1 — Observe: coverage and portfolio understanding

### Tasks
- Confirm the grain and exposure.
- Compare environmental stress flags with insured trigger flags.
- Summarize exposure-adjusted claim frequency and paid severity by region and occupancy.
- Create at least one useful visualization.
- Write 3–5 sentences explaining why environmental deterioration is not the same as an insured claim.

In [ ]:
# TODO: Build a compact portfolio summary.
# Suggested checks:
# - exposure by year
# - claim count / exposure
# - rows with air/water stress vs rows with air/water insured trigger
# - aggregate paid loss by region and occupancy

# YOUR CODE HERE


In [ ]:
# TODO: Create at least one visualization that helps explain baseline portfolio risk.

# YOUR CODE HERE


**TODO — Interpretation:**  
Write your coverage/hazard-to-claim interpretation here.

## Stage 2 — Model: exposure-adjusted claim frequency

Fit a count model for `claim_count`. Use `log(exposure_years)` as an offset.

A useful baseline specification may include:
- occupancy and region;
- air and water environmental variables;
- climate variables;
- air/water stress flags and their interaction;
- mitigation indicators.

Interpret selected coefficients as **rate ratios** using `exp(coefficient)`.

In [ ]:
# TODO: Fit an exposure-adjusted Poisson GLM.
# Example structure:
#
# freq_formula = "claim_count ~ C(occupancy) + C(region) + ..."
# freq_model = smf.glm(
#     formula=freq_formula,
#     data=dfm,
#     family=sm.families.Poisson(),
#     offset=np.log(dfm["exposure_years"])
# ).fit()
#
# display(freq_model.summary())

# YOUR CODE HERE


In [ ]:
# TODO: Convert selected coefficients to rate ratios and interpret them.

# YOUR CODE HERE


## Stage 3 — Model: conditional severity

For rows with claims, define:

`average_paid_severity = aggregate_paid_loss / claim_count`

Fit a positive severity model. A Gamma GLM with log link is a reasonable baseline. Consider weighting the average severity observation by `claim_count` because a row representing multiple claims contains more severity information than a row with one claim.

In [ ]:
# TODO: Build the claim-positive severity dataset and fit a conditional severity model.

# YOUR CODE HERE


**TODO — Frequency vs severity interpretation:**  
Which variables appear to affect frequency and severity differently?

## Stage 4 — Price: expected loss and premium adequacy

For the 2026 current portfolio:

1. predict expected claim count;
2. predict conditional paid severity;
3. calculate expected loss;
4. calculate expected loss cost per exposure;
5. calculate indicated premium;
6. compare with current premium.

Use:

`Indicated Premium = (Expected Loss + 750 × Exposure) / 0.70`

`Adequacy Ratio = Current Premium / Indicated Premium`

In [ ]:
# TODO: Produce baseline 2026 expected loss and premium adequacy.
# You will need your fitted `freq_model` and `sev_model`.

# YOUR CODE HERE


## Stage 5 — Stress: four scenarios and tail risk

Required scenarios:
- Baseline
- Air stress
- Water stress
- Compound stress

Use the provided `apply_scenario()` helper. Recompute model predictions for each scenario.

Then simulate annual aggregate paid loss and calculate:
- expected annual loss;
- indicated premium;
- adequacy ratio;
- VaR 99%;
- TVaR 99%.

Use a fixed seed.

In [ ]:
# TODO: Write a reusable function that:
# 1. accepts a scenario portfolio,
# 2. predicts claim count and severity,
# 3. simulates annual aggregate paid loss,
# 4. returns expected loss, indicated premium, adequacy, VaR99 and TVaR99.
#
# Hint: for a compound Poisson-Gamma approximation, if N ~ Poisson(mu_n)
# and individual severity is Gamma(shape=k, scale=mu_s/k), then conditional
# on N=n the aggregate severity is Gamma(shape=n*k, scale=mu_s/k).

# YOUR CODE HERE


In [ ]:
# TODO: Run all four scenarios and create a comparison table.

SCENARIOS = ["Baseline", "Air stress", "Water stress", "Compound stress"]

# YOUR CODE HERE


In [ ]:
# TODO: Plot the four simulated aggregate annual loss distributions,
# or another tail-risk visualization that clearly compares scenarios.

# YOUR CODE HERE


## Stage 6 — Mitigate: constrained resilience decision

Budget: **8,000,000 currency units**

Available actions where absent:
- air filtration;
- water treatment;
- business continuity plan.

Construct a manageable set of feasible candidate mitigation portfolios. Examples:
- air-focused;
- water-focused;
- BCP-focused;
- balanced expected-loss-reduction-per-cost;
- tail-focused/compound-risk-focused.

For each candidate portfolio:
1. keep total implementation cost ≤ budget;
2. modify the mitigation indicators for selected locations;
3. re-predict the **compound stress** portfolio;
4. simulate aggregate annual loss;
5. calculate expected loss, VaR99, TVaR99 and post-mitigation adequacy.

Choose the feasible portfolio with the **lowest TVaR99**.

Report expected-loss reduction and benefit-cost ratio as supporting measures, not as the sole decision rule.

In [ ]:
# TODO: Build candidate mitigation actions from the 2026 portfolio.
# Each action should identify:
# - location_id
# - action type
# - cost
# - the mitigation indicator that would change from 0 to 1

# YOUR CODE HERE


In [ ]:
# TODO: Construct and evaluate feasible mitigation portfolios under the budget.
# Select the candidate portfolio with the lowest TVaR99.

# YOUR CODE HERE


## Final Climate Risk Committee recommendation

Complete this section after your analysis.

### 1. Baseline position
- Expected annual loss:
- Indicated premium:
- Premium adequacy ratio:
- VaR99:
- TVaR99:

### 2. Most material climate stress
State which scenario is most material and quantify the change.

### 3. Mitigation decision
State the selected portfolio, cost, expected-loss reduction, BCR, VaR reduction, and TVaR reduction.

### 4. Pricing / underwriting recommendation
Choose and defend one or more: **Maintain / Reprice / Mitigate / Restrict / Monitor**.

### 5. Limitations
State at least three assumptions or limitations.
